# Ceteris Paribus on Aurora Small with ERA5

This notebook loads the public **Aurora 0.25° Small** checkpoint, builds a batch from one day of
ERA5 the way Aurora's own documentation does, and sends a Ceteris Paribus probe through it with
xsonde: specific humidity is shifted by whole standard deviations and the response of the
2 m temperature and mean sea-level pressure forecasts is read back.

Requirements:

```bash
pip install "xsonde[aurora]" cdsapi matplotlib
```

ERA5 comes from the Copernicus Climate Data Store, which needs an account and a `~/.cdsapirc`
file with your key. The first run downloads about 250 MB into `examples/data/`. A GPU makes the
seven forward passes take seconds; on a CPU expect a few minutes.

In [ ]:
import sys
from datetime import datetime

import numpy as np
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
from era5_loader import download, load_batch
from xsonde import METHODS, results
from xsonde.adapters import AuroraAdapter

device = "cuda" if torch.cuda.is_available() else "cpu"
START = datetime(2023, 1, 1, 0)
device

## Data

`download` fetches the static fields (geopotential, land-sea mask, soil type), the four surface
variables and the five atmospheric variables on Aurora's 13 pressure levels for one day, at
00, 06, 12 and 18 UTC. Files that already exist are skipped.

In [ ]:
paths = download(START)
paths

## Model and adapter

The adapter is the only thing xsonde needs to know about Aurora. `load_batch` is the loader from
`era5_loader.py`: it returns an Aurora `Batch` whose two input steps are `START` and six hours
later, so the forecast is valid twelve hours after `START`.

In [ ]:
try:
    from aurora import AuroraSmallPretrained as AuroraSmall
except ImportError:
    from aurora import AuroraSmall

model = AuroraSmall()
model.load_checkpoint("microsoft/aurora", "aurora-0.25-small-pretrained.ckpt")
model = model.to(device).eval()

adapter = AuroraAdapter(model=model, loader=load_batch, device=device)
batch = adapter.load_batch(START)
{k: tuple(v.shape) for k, v in batch.surf_vars.items()}

## Unperturbed forecast

A sanity check before probing: the adapter's `predict` returns a dict of physical fields.

In [ ]:
with torch.inference_mode():
    pred = adapter.predict(batch)

t2 = pred["2t"][0, 0].cpu().numpy() - 273.15
plt.figure(figsize=(11, 4.5))
plt.imshow(t2, cmap="RdBu_r", vmin=-40, vmax=40)
plt.colorbar(label="2 m temperature [°C]", shrink=0.8)
plt.title("Aurora Small, 6 h forecast")
plt.axis("off")
plt.show()

## The probe

A probe is a source, a list of targets and a list of reductions. Here the source is specific
humidity at every level, shifted by −3σ to +3σ in the model's normalised space (`mode: dist`),
and each target field is reduced to a global mean, a mean over Europe, the RMS change against
the unperturbed forecast, and its spatial spread.

In [ ]:
EUROPE = {"h0": 120, "h1": 200, "w0": 0, "w1": 120}

probe = METHODS["ceteris_paribus"]({
    "source": "q",
    "targets": ["2t", "msl"],
    "mode": "dist",
    "perturbations": [-3, -2, -1, 0, 1, 2, 3],
    "reductions": [
        {"name": "global_mean", "label": "globalMean"},
        {"name": "regional_mean", "label": "europe", **EUROPE},
        {"name": "rmse_response", "label": "rmseResp"},
        {"name": "spatial_std", "label": "spatialStd"},
    ],
})
result = probe.run(adapter, batch)
result.results["2t"]["rmseResp"]

## Response curves

The thesis convention: the RMS response of each target, divided by the spatial spread of the
unperturbed forecast, so the two targets are comparable.

In [ ]:
deltas = result.meta["perturbations"]
base = deltas.index(0)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, target in zip(axes, probe.cfg.targets):
    r = result.results[target]
    ax.plot(deltas, np.array(r["rmseResp"]) / r["spatialStd"][base], marker="o")
    ax.set_title(f"target: {target}")
    ax.set_xlabel("perturbation of q (σ units)")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("RMS response / spatial spread")
plt.show()

## Response map

`result.fields` holds the full forecast field for every perturbation, so the difference between
the +1σ and the unperturbed forecast shows where the model moves.

In [ ]:
diff = result.fields["2t"][deltas.index(1)] - result.fields["2t"][base]
lim = np.percentile(np.abs(diff), 99)

plt.figure(figsize=(11, 4.5))
plt.imshow(diff, cmap="RdBu_r", vmin=-lim, vmax=lim)
plt.colorbar(label="Δ 2 m temperature [K] at +1σ humidity", shrink=0.8)
plt.title("Response of the 2 m temperature forecast")
plt.axis("off")
plt.show()

## Saving

Results are written as netCDF plus JSON and come back as the same `Result` object, or as xarray
datasets for your own analysis.

In [ ]:
out = "runs/cp_q_2t_msl"
results.save(result, out)
loaded = results.load(out)
loaded.to_datasets()["2t"]

## From a notebook to a sweep

The same probe as a yaml file is in `sweeps/aurora_small_era5.yaml`. It declares the adapter,
the date and the method once, and a `sweep` block expands it over sources and dates:

```bash
PYTHONPATH=. xsonde run sweeps/aurora_small_era5.yaml
```

Inside a SLURM job with several tasks the runs are sharded across tasks automatically, and runs
that already carry a `done` marker are skipped on resubmission.